In [4]:
import os
import glob
import json
import faiss
import torch
import clip
import numpy as np
from PIL import Image

# =========================
# CONFIG
# =========================
FRAME_FOLDER = "data/video_02"
INDEX_PATH = "data/bin/l14_video02.bin"
MAPPING_PATH = "data/bin/l14_video02_metadata.json"

VIDEO_NAME = "video_02.mp4"
FPS = 1  # quan trọng để tính timestamp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD MODEL
# =========================
model, preprocess = clip.load("ViT-L/14", device=DEVICE)
model.eval()

# =========================
# LOAD FRAMES
# =========================
frame_paths = sorted(glob.glob(os.path.join(FRAME_FOLDER, "*.*")))

print(f"Found {len(frame_paths)} frames")

# =========================
# EXTRACT EMBEDDINGS
# =========================
batch_size = 32
embeddings = []
metadata = []

with torch.no_grad():
    for i in range(0, len(frame_paths), batch_size):
        batch_paths = frame_paths[i:i + batch_size]

        images = []
        batch_meta = []

        for j, p in enumerate(batch_paths):
            img = preprocess(Image.open(p).convert("RGB"))
            images.append(img)

            frame_id = i + j
            timestamp = frame_id / FPS

            batch_meta.append({
                "frame": p.replace("\\", "/"),
                "video": VIDEO_NAME,
                "timestamp": round(timestamp, 2)
            })

        images = torch.stack(images).to(DEVICE)

        feats = model.encode_image(images)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        embeddings.append(feats.cpu().numpy().astype("float32"))
        metadata.extend(batch_meta)

        print(f"Processed {min(i + batch_size, len(frame_paths))}/{len(frame_paths)}")

# =========================
# CONCAT EMBEDDINGS
# =========================
embeddings = np.vstack(embeddings)

print("Embedding shape:", embeddings.shape)

# =========================
# BUILD FAISS INDEX
# =========================
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings)

# =========================
# SAVE OUTPUTS
# =========================
os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)

faiss.write_index(index, INDEX_PATH)

with open(MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved FAISS index:", INDEX_PATH)
print("Saved metadata:", MAPPING_PATH)

Found 1329 frames
Processed 32/1329
Processed 64/1329
Processed 96/1329
Processed 128/1329
Processed 160/1329
Processed 192/1329
Processed 224/1329
Processed 256/1329
Processed 288/1329
Processed 320/1329
Processed 352/1329
Processed 384/1329
Processed 416/1329
Processed 448/1329
Processed 480/1329
Processed 512/1329
Processed 544/1329
Processed 576/1329
Processed 608/1329
Processed 640/1329
Processed 672/1329
Processed 704/1329
Processed 736/1329
Processed 768/1329
Processed 800/1329
Processed 832/1329
Processed 864/1329
Processed 896/1329
Processed 928/1329
Processed 960/1329
Processed 992/1329
Processed 1024/1329
Processed 1056/1329
Processed 1088/1329
Processed 1120/1329
Processed 1152/1329
Processed 1184/1329
Processed 1216/1329
Processed 1248/1329
Processed 1280/1329
Processed 1312/1329
Processed 1329/1329
Embedding shape: (1329, 768)
Saved FAISS index: data/bin/l14_video02.bin
Saved metadata: data/bin/l14_video02_metadata.json


In [ ]:
#ocr

In [5]:
import os
import glob
import json
import cv2
import easyocr

# =========================
# CONFIG
# =========================
FRAME_FOLDER = "data/video_01"
OUTPUT_JSON = "data/bin/video01_ocr.json"

VIDEO_NAME = "video_01.mp4"
FPS = 1

# =========================
# LOAD OCR MODEL
# =========================
reader = easyocr.Reader(
    ['vi', 'en'],
    gpu=True   # đổi False nếu không có GPU
)

# =========================
# LOAD FRAMES
# =========================
frame_paths = sorted(glob.glob(os.path.join(FRAME_FOLDER, "*.*")))

print(f"Found {len(frame_paths)} frames")

ocr_metadata = []

# =========================
# FUNCTION PREPROCESS
# =========================
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    # grayscale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # reduce noise
    img = cv2.GaussianBlur(img, (3, 3), 0)
    # increase contrast
    img = cv2.convertScaleAbs(img, alpha=1.5, beta=10)
    return img
# =========================
# OCR LOOP
# =========================
for i, frame_path in enumerate(frame_paths):
    img = preprocess_image(frame_path)
    if img is None:
        print(f"Skip frame: {frame_path}")
        continue
    result = reader.readtext(
        img,
        detail=1,
        paragraph=True,
        decoder='beamsearch',
        beamWidth=10,
        contrast_ths=0.1,
        adjust_contrast=0.7
    )

    texts = [r[1] for r in result]
    ocr_text = " ".join(texts)

    timestamp = round(i / FPS, 2)

    ocr_metadata.append({
        "frame": frame_path.replace("\\", "/"),
        "video": VIDEO_NAME,
        "timestamp": timestamp,
        "ocr": ocr_text
    })

    print(f"[{i+1}/{len(frame_paths)}]")
    print("OCR:", ocr_text)

# =========================
# SAVE JSON
# =========================
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(ocr_metadata, f, ensure_ascii=False, indent=2)

print("Done.")

Found 1341 frames
[1/1341]
OCR: 
[2/1341]
OCR: TIMTÚC
[3/1341]
OCR: Cà Mau: Động thổ Dự án Nhà máy điện gió Nhật Bản Bạc Liêu gẩn 2.500 tỷ dông
[4/1341]
OCR: TUC Bolivia ban bố tình trạng khẩn cấp do gián doạn giao thông duờng bộ Thái Lan
[5/1341]
OCR: TIn TUc Bolivia ban bố tình trạng khẩn cấp do gián doạn giao thông dường bộ Thái Lan phá dườne
[6/1341]
OCR: Tin TỨC an bố tình trạng khẩn cấp do gián doạn giao thông duờng bộ Thái Lan phá duờng dây cà bc
[7/1341]
OCR: rạng khẩn cấp do gián doạn giao thông dường bộ Thái Lan phá dường dôy cà bạc trực tuyế
[8/1341]
OCR: cấp do gián doạn giao thông duờng bộ Thái Lan phá dường dây cà bạc trực tuyến giao dịct Gazs
[9/1341]
OCR: Gazti In doạn giao thông duờng bộ Thái Lan phá dường dôy cờ bạc trực tuyến giao dịch hàng triệL
[10/1341]
OCR: Gzri 10 thông duờng bộ Thái Lan phá dường dây cà bạc trực tuyến giao dich hàng triệu USD mỗi
[11/1341]
OCR: rờng bộ Thái Lan phá duờng dôy cà bạc trực tuyến giao dịch hàng triệu USD mỗi tháng 7
[12/1341]
OCR: 

In [ ]:
#audio 

In [4]:
import os
import subprocess
import json
from faster_whisper import WhisperModel

VIDEO_FOLDER = "video"
AUDIO_FOLDER = "data/audio"
OUTPUT_CLIPS = "data/clips"
TRANSCRIPT_FILE = "data/transcript.json"

os.makedirs(AUDIO_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_CLIPS, exist_ok=True)


# ======================
# STEP 1: EXTRACT AUDIO
# ======================
def extract_audio():
    for file in os.listdir(VIDEO_FOLDER):
        if not file.endswith(".mp4"):
            continue

        video_path = os.path.join(VIDEO_FOLDER, file)
        wav_path = os.path.join(AUDIO_FOLDER, file.replace(".mp4", ".wav"))

        cmd = [
            "ffmpeg",
            "-y",
            "-i", video_path,
            "-vn",
            "-ac", "1",
            "-ar", "16000",
            wav_path
        ]

        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print("🎧 Audio extraction done")


# ======================
# STEP 2: TRANSCRIBE + CUT
# ======================
def transcribe_and_cut():
    model = WhisperModel(
        "small",
        device="cuda",
        compute_type="float16"
    )

    results = []

    for file in os.listdir(AUDIO_FOLDER):
        if not file.endswith(".wav"):
            continue

        audio_path = os.path.join(AUDIO_FOLDER, file)
        video_name = file.replace(".wav", "")
        video_path = os.path.join(VIDEO_FOLDER, video_name + ".mp4")

        segments, info = model.transcribe(
            audio_path,
            language="vi",
            beam_size=5
        )

        for i, seg in enumerate(segments):

            start = float(seg.start)
            end = float(seg.end)
            text = seg.text.strip()

            clip_name = f"{video_name}_seg_{i}.mp4"
            clip_path = os.path.join(OUTPUT_CLIPS, clip_name)

            cmd = [
                "ffmpeg",
                "-y",
                "-ss", str(start),
                "-to", str(end),
                "-i", video_path,
                "-c", "copy",
                clip_path
            ]

            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            results.append({
                "video": video_name,
                "clip_path": clip_path,
                "start": start,
                "end": end,
                "text": text
            })

    with open(TRANSCRIPT_FILE, "w", encoding="utf8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print("🔥 Done")


# ======================
# RUN
# ======================
if __name__ == "__main__":
    extract_audio()
    transcribe_and_cut()

🎧 Audio extraction done
🔥 Done


In [10]:
import re
from collections import defaultdict

class HashOCRIndex:
    def __init__(self):
        # word → set(frame)
        self.index = defaultdict(set)

        # lưu metadata để trả kết quả đầy đủ
        self.meta = {}

    # -------------------------
    # CLEAN TEXT
    # -------------------------
    def clean(self, text):
        text = text.lower()
        text = re.sub(r"[^a-zA-Z0-9À-ỹ\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # -------------------------
    # ADD DOCUMENT
    # -------------------------
    def add(self, item):
        """
        item = {
            frame, video, timestamp, ocr
        }
        """

        frame = item["frame"]

        # lưu metadata
        self.meta[frame] = item

        # clean OCR
        text = self.clean(item["ocr"])
        words = text.split()

        # build inverted hash index
        for w in words:
            self.index[w].add(frame)

    # -------------------------
    # BUILD FROM JSON
    # -------------------------
    def build(self, data):
        for item in data:
            self.add(item)

    # -------------------------
    # SEARCH
    # -------------------------
    def search(self, query):
        query = self.clean(query)
        words = query.split()

        if not words:
            return []

        # lấy set giao nhau
        result = None

        for w in words:
            frames = self.index.get(w, set())

            if result is None:
                result = frames
            else:
                result = result & frames  # INTERSECTION

        if not result:
            return []

        # trả full metadata
        return [self.meta[f] for f in result]

In [11]:
import json

# load file OCR
with open("data/bin/video01_ocr.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# init index
index = HashOCRIndex()

# build index
index.build(data)

# search


In [22]:
import time

# giả sử bạn đã có index
# index = HashOCRIndex()
# index.build(data)

def benchmark_search(index, query, repeat=1000):
    start = time.perf_counter()

    for _ in range(repeat):
        results = index.search(query)

    end = time.perf_counter()

    total_time = end - start
    avg_time = total_time / repeat

    print("🔍 Query:", query)
    print("📦 Total runs:", repeat)
    print("⏱ Total time:", total_time, "seconds")
    print("⚡ Avg time per search:", avg_time * 1000, "ms")
    print("📊 Results sample:", results[:3])
    
benchmark_search(index, "tin tức", repeat=1000)

🔍 Query: tin tức
📦 Total runs: 1000
⏱ Total time: 0.008776399998168927 seconds
⚡ Avg time per search: 0.008776399998168927 ms
📊 Results sample: [{'frame': 'data/video_01/frame_1336.webp', 'video': 'video_01.mp4', 'timestamp': 375.0, 'ocr': 'Tin TÚc TRUNG TÂM TIN TỨc'}, {'frame': 'data/video_01/frame_826.webp', 'video': 'video_01.mp4', 'timestamp': 1149.0, 'ocr': '54T Tin TỨC Jộng thổ Dự án Nhà máy điện gió Nhật Bản Bạc Liêu gẩn 2.500 tỷ dông Đắk Lắk xây dựng'}, {'frame': 'data/video_01/frame_1339.webp', 'video': 'video_01.mp4', 'timestamp': 378.0, 'ocr': 'Tin TÚc TRUNG TÂM TIN TỨC ĐÀl PHÁT THANH VÀ TRUYÊN HÌNH THÀNH PHỐ HỔ CHÍ MINH'}]


In [24]:
import json
import time

# -------------------------
# LOAD JSON OCR FILE
# -------------------------
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# -------------------------
# O(N SEARCH)
# -------------------------
def search_o_n(data, query):
    query = query.lower()

    results = []

    for item in data:
        # scan từng frame (O(n))
        if query in item["ocr"].lower():
            results.append(item)

    return results


# -------------------------
# BENCHMARK FUNCTION
# -------------------------
def benchmark_o_n(data, query, repeat=1000):
    start = time.perf_counter()

    for _ in range(repeat):
        results = search_o_n(data, query)

    end = time.perf_counter()

    total_time = end - start
    avg_time = total_time / repeat

    print("🔍 QUERY:", query)
    print("📦 TOTAL RUNS:", repeat)
    print("⏱ TOTAL TIME:", total_time, "seconds")
    print("⚡ AVG TIME:", avg_time * 1000, "ms")
    print("📊 SAMPLE RESULT:", results[:2])

In [25]:
data = load_data("data/bin/video01_ocr.json")

benchmark_o_n(data, "tin tức", repeat=1000)

🔍 QUERY: tin tức
📦 TOTAL RUNS: 1000
⏱ TOTAL TIME: 1.879185800000414 seconds
⚡ AVG TIME: 1.879185800000414 ms
📊 SAMPLE RESULT: [{'frame': 'data/video_01/frame_1002.webp', 'video': 'video_01.mp4', 'timestamp': 5.0, 'ocr': 'Tin TỨC an bố tình trạng khẩn cấp do gián doạn giao thông duờng bộ Thái Lan phá duờng dây cà bc'}, {'frame': 'data/video_01/frame_1336.webp', 'video': 'video_01.mp4', 'timestamp': 375.0, 'ocr': 'Tin TÚc TRUNG TÂM TIN TỨc'}]
